In [0]:
USE CATALOG meridian_t02;

SELECT 
    mv.user_country,
    mv.mv_active_users,
    raw.raw_active_users,
    mv.mv_sessions,
    raw.raw_sessions,
    mv.mv_conversion_pct,
    raw.raw_conversion_pct
FROM (
    SELECT 
        user_country,
        MEASURE(active_users) AS mv_active_users,
        MEASURE(total_sessions) AS mv_sessions,
        ROUND(MEASURE(search_to_booking_conversion) * 100, 2) AS mv_conversion_pct
    FROM gold.mv_stays_kpis
    GROUP BY user_country
) mv
JOIN (
    SELECT 
        user_country,
        COUNT(DISTINCT user_id) AS raw_active_users,
        COUNT(DISTINCT session_id) AS raw_sessions,
        ROUND((COUNT(DISTINCT CASE WHEN is_booked = 1 THEN session_id END) / COUNT(DISTINCT session_id)) * 100, 2) AS raw_conversion_pct
    FROM gold.fct_sessions
    GROUP BY user_country
) raw ON mv.user_country = raw.user_country
ORDER BY mv.mv_sessions DESC
LIMIT 5;